# ALLO validation scaffold

Reproducible scaffold for `MODEL_CARD.md` §7-§8. Labels are treated as presence-only, biased, and incomplete. This notebook intentionally avoids naive accuracy and random splits. The minimum comparison baseline is Tier 0 A25.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
ENGINE = ROOT / "eco-stochast-poc" / "python_engine"
if str(ENGINE) not in sys.path:
    sys.path.insert(0, str(ENGINE))

SEED = 20260701

## Load synthetic data through Stage 5 loaders

Until verified IDEA/SIMAC rainfall and SIMMA exports are wired in, this notebook uses deterministic synthetic rainfall. Synthetic labels only exercise the protocol; they are not calibration evidence.

In [ ]:
from validation.scaffold import (
    build_synthetic_presence_only_dataset,
    temporal_train_test_split,
    rainfall_event_backtest_windows,
    build_validation_report,
)

examples, a25_records = build_synthetic_presence_only_dataset(seed=SEED, days=1100)
len(examples), examples[0], examples[-1]

## Temporal split only

Rainfall has memory and adjacent days share hydrologic context. Random splits leak future rainfall/event information into training, so this scaffold only allows a time-based split.

In [ ]:
split = temporal_train_test_split(examples, test_start_year=2028)
{
    "train_years": split.train_years,
    "test_years": split.test_years,
    "n_train": len(split.train),
    "n_test": len(split.test),
    "warning": split.warning,
}

## Event-based rainfall backtesting

Backtesting units are rainfall episodes, not shuffled daily rows. This matches the event-based validation requirement in the model card.

In [ ]:
labels_by_day = {example.day: example.observed_label for example in examples}
windows = rainfall_event_backtest_windows(a25_records, labels_by_day, event_threshold_mm=200.0)
len(windows), windows[:3]

## Metrics and A25 comparison

Reported metrics: precision/recall, PR-AUC, Brier score, calibration curve, lead time, false-alarm rate, missed-event rate, per-threshold alert evaluation, and uncertainty coverage. Naive accuracy is omitted because unlabeled windows are not confirmed negatives.

In [ ]:
test_labels = {example.day: example.observed_label for example in split.test}
test_windows = [
    window for window in windows
    if window.start_day.year in split.test_years or window.end_day.year in split.test_years
]
report = build_validation_report(split.test, test_windows)
report

## Interpretation guardrails

- Presence-only labels mean false negatives are structurally uncertain.
- A result that does not beat A25 is not evidence of model improvement.
- Synthetic runs are smoke tests for the protocol, not scientific validation.